# 10 — Build the training set

**Purpose.** Pairs each glacier's chosen training scene with its DEM and mask, and writes the
model-ready set (`glacierview.training.build_training_pair`): the six `COMMON_BANDS` and the DEM,
normalised and resized to 128×128 and stacked into 7 channels, plus the mask resized to match. Glaciers
without a mask are skipped. NDSI and NDWI are added at train time, giving the model's 9 channels.

## Inputs
- **`filtered_training_data_summer_months.csv`** in
  `config.processed_metadata_dir(config.TRAINING_DATA_LABEL)`: the saved Athena result of
  `sql/training_data_query.sql`. One summer scene per glacier, 7,174 rows. See notebook 06.
- **Training scenes and DEMs**: `config.landsat_dir(...)` and `config.dem_dir(...)` for
  `config.TRAINING_DATA_LABEL`, from notebooks 03 and 04. The training DEMs on disk have no S3 copy.
- **Masks**: `config.masks_dir()`, from notebook 09.

## Outputs
`config.training_set_dir()` (`src/segmentation/training/data/processed_training_data_summer_months/`,
gitignored), the default `glacierview train --data-dir`:
- `images/<file_name>.tif`: 128×128×7 float stacks, one per training scene (7,174 locally).
- `masks/<glims_id>.tif`: 128×128 masks.

The canonical copy is `s3://training-images-t1-l2-sr` (us-west-1), `images/` + `masks/`.

## Open to-do
- Identify the right thermal band for Landsat 8.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from tqdm import tqdm

from glacierview import config
from glacierview.training import build_training_pair

LABEL = config.TRAINING_DATA_LABEL
chosen = pd.read_csv(config.processed_metadata_dir(LABEL)
                     / "filtered_training_data_summer_months.csv")
out = config.training_set_dir()
for sub in ("images", "masks"):
    (out / sub).mkdir(parents=True, exist_ok=True)

In [ ]:
for row in tqdm(chosen.itertuples(), total=len(chosen)):
    mask_path = config.masks_dir() / f"{row.glims_id}.tif"
    if not mask_path.exists():
        continue
    image, label = build_training_pair(
        config.glacier_dir(row.glims_id, LABEL) / row.file_name,
        config.dem_path(row.glims_id, LABEL),
        mask_path,
    )
    tifffile.imwrite(out / "images" / row.file_name, image, planarconfig="contig")
    tifffile.imwrite(out / "masks" / f"{row.glims_id}.tif", label)

## Inspect a few pairs

In [ ]:
channels = [*config.COMMON_BANDS, "dem", "mask"]
pairs = sorted((out / "images").glob("*.tif"))
for path in random.sample(pairs, min(4, len(pairs))):
    image = tifffile.imread(path)[0]
    label = tifffile.imread(out / "masks" / f"{path.name.split('_')[0]}.tif")
    label = label.reshape(*image.shape[:2], 1)
    stack = np.concatenate([image, label], axis=2)
    fig, axs = plt.subplots(2, 4, figsize=(10, 5))
    fig.suptitle(path.name)
    for ax, name, channel in zip(axs.flat, channels, np.moveaxis(stack, -1, 0), strict=True):
        ax.imshow(channel)
        ax.set_title(name)
        ax.axis("off")
    plt.show()